<a href="https://colab.research.google.com/github/asfiya-tehmeen/automobile-sales-recession-analysis/blob/main/Automobile_Sales_Dashboard_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyzing the Impact of Recession on Automobile Sales
## Part 2: Create Dashboard using Plotly and Dash

In this notebook we build an interactive Dash dashboard so the directors of XYZAutomotives can
drill into the automobile sales data by **Recession Period** or by **Year**.

**Tasks covered:**
- TASK 2.1: Create a Dash application and give it a meaningful title.
- TASK 2.2: Add drop-down menus with appropriate titles and options.
- TASK 2.3: Add a division for output display with appropriate id and classname.
- TASK 2.4: Create callback(s) to update the input container and output container.
- TASK 2.5: Create and display graphs for Recession Report Statistics.
- TASK 2.6: Create and display graphs for Yearly Report Statistics.

## 0. Install / import required libraries

In [7]:
!pip install dash pandas plotly --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 35.4 MB/s eta 0:00:00


In [8]:
# Run this once if the packages are not already installed
# !pip install dash jupyter-dash pandas plotly --quiet

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from dash import Dash, dcc, html, Input, Output

## 1. Load the data

The dataset used for this assignment is the IBM-provided synthetic "Historical Automobile Sales"
dataset. We load it directly from the URL used throughout the course. If you have your own local
copy (e.g. `automobile_sales.csv` in the same folder as this notebook), feel free to point
`data_path` at that file instead — the rest of the notebook works unchanged as long as the same
column names are present (`Date`, `Recession`, `Automobile_Sales`, `GDP`, `Unemployment_Rate`,
`Consumer_Confidence`, `Seasonality_Weight`, `Price`, `Advertising_Expenditure`, `Vehicle_Type`,
`Competition`, `Month`, `Year`).

In [9]:
data_path = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv'

try:
    df = pd.read_csv(data_path)
    print("Loaded data from remote URL.")
except Exception as e:
    print("Could not load remote dataset, generating a synthetic fallback dataset instead.")
    print("Reason:", e)

    rng = np.random.default_rng(42)
    dates = pd.date_range('1980-01-31', '2023-12-31', freq='M')

    recession_periods = [
        ('1980-01-01', '1980-12-31'),
        ('1981-01-01', '1982-12-31'),
        ('1991-01-01', '1991-12-31'),
        ('2000-01-01', '2001-12-31'),
        ('2007-12-01', '2009-06-30'),
        ('2020-02-01', '2020-04-30'),
    ]

    def in_recession(d):
        for start, end in recession_periods:
            if pd.Timestamp(start) <= d <= pd.Timestamp(end):
                return 1
        return 0

    vehicle_types = ['Supperminicar', 'Smallfamiliycar', 'Mediumfamilycar', 'Executivecar', 'Sports']

    rows = []
    for d in dates:
        rec = in_recession(d)
        for vt in vehicle_types:
            base_sales = rng.normal(800, 100)
            sales = max(base_sales - (rec * rng.normal(250, 50)), 50)
            rows.append({
                'Date': d,
                'Year': d.year,
                'Month': d.strftime('%b'),
                'Recession': rec,
                'Automobile_Sales': round(sales, 0),
                'GDP': round(rng.normal(45000, 5000) - rec * 3000, 2),
                'Unemployment_Rate': round(rng.normal(6, 1) + rec * 2, 2),
                'Consumer_Confidence': round(rng.normal(100, 10) - rec * 20, 2),
                'Seasonality_Weight': round(rng.normal(1, 0.15), 2),
                'Price': round(rng.normal(28000, 3000), 2),
                'Advertising_Expenditure': round(rng.normal(15000, 3000) - rec * 2000, 2),
                'Vehicle_Type': vt,
                'Competition': round(rng.uniform(1, 10), 2),
            })
    df = pd.DataFrame(rows)

df['Date'] = pd.to_datetime(df['Date'])
if 'Year' not in df.columns:
    df['Year'] = df['Date'].dt.year
if 'Month' not in df.columns:
    df['Month'] = df['Date'].dt.strftime('%b')

df.head()

Loaded data from remote URL.


,Date,Year,Month,Recession,Consumer_Confidence,Seasonality_Weight,Price,Advertising_Expenditure,Competition,GDP,Growth_Rate,unemployment_rate,Automobile_Sales,Vehicle_Type,City
0,1980-01-31,1980,Jan,1,108.24,0.50,27483.571,1558,7,60.223,0.010000,5.4,456.0,Supperminicar,Georgia
1,1980-02-29,1980,Feb,1,98.75,0.75,24308.678,3048,4,45.986,-0.309594,4.8,555.9,Supperminicar,New York
2,1980-03-31,1980,Mar,1,107.48,0.20,28238.443,3137,3,35.141,-0.308614,3.4,620.0,Mediumfamilycar,New York
3,1980-04-30,1980,Apr,1,115.01,1.00,32615.149,1653,7,45.673,0.230596,4.2,702.8,Supperminicar,Illinois
4,1980-05-31,1980,May,1,98.72,0.20,23829.233,1319,4,52.997,0.138197,5.3,770.4,Smallfamiliycar,California


In [10]:
year_list = sorted(df['Year'].unique().tolist())
print("Years available:", year_list[:5], '...', year_list[-5:])
print("Vehicle types:", df['Vehicle_Type'].unique().tolist())
df.info()


Years available: [1980, 1981, 1982, 1983, 1984] ... [2019, 2020, 2021, 2022, 2023]
Vehicle types: ['Supperminicar', 'Mediumfamilycar', 'Smallfamiliycar', 'Sports', 'Executivecar']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 528 entries, 0 to 527
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Date                     528 non-null    datetime64[ns]
 1   Year                     528 non-null    int64         
 2   Month                    528 non-null    object        
 3   Recession                528 non-null    int64         
 4   Consumer_Confidence      528 non-null    float64       
 5   Seasonality_Weight       528 non-null    float64       
 6   Price                    528 non-null    float64       
 7   Advertising_Expenditure  528 non-null    int64         
 8   Competition              528 non-null    int64         
 9   GDP                      528 non-null  

## 2. TASK 2.1 — Create the Dash application

We create a `Dash` app instance and give the page a meaningful title,
`"XYZAutomotives Sales Dashboard"`, both in the browser tab (`app.title`) and as an `<H1>`
heading at the top of the layout.

In [11]:
app = Dash(__name__)
app.title = "XYZAutomotives Sales Dashboard"

## 3. TASK 2.2 — Dropdown menus

We add two dropdown menus:

1. **Select Statistics** — lets the director choose between `Recession Period Statistics` and
   `Yearly Statistics`.
2. **Select Year** — lets the director choose a specific year. This dropdown is only meaningful
   (and only enabled) when "Yearly Statistics" is selected — this is handled in the callback in
   TASK 2.4.

In [12]:
dropdown_statistics = dcc.Dropdown(
    id='dropdown-statistics',
    options=[
        {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'},
        {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
    ],
    value='Recession Period Statistics',
    placeholder='Select a report type',
    style={'width': '80%', 'padding': '3px', 'textAlign': 'center'}
)

dropdown_year = dcc.Dropdown(
    id='select-year',
    options=[{'label': str(y), 'value': y} for y in year_list],
    value=year_list[-1],
    placeholder='Select a year',
    style={'width': '80%', 'padding': '3px', 'textAlign': 'center'}
)


## 4. TASK 2.3 — Output display division

We define an output container `<div>` with `id='output-container'` and
`className='chart-grid'`. The four resulting charts (for either report type) are placed inside
this single container as two stacked rows, each containing two graphs.

In [13]:
app.layout = html.Div([

    html.H1(
        "XYZAutomotives Sales Dashboard",
        style={'textAlign': 'center', 'color': '#503D36', 'fontSize': 26}
    ),

    html.Div([
        html.Div([
            html.Label("Select Statistics:"),
            dropdown_statistics,
        ], style={'width': '48%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Year:"),
            dropdown_year,
        ], style={'width': '48%', 'display': 'inline-block'}),
    ], style={'display': 'flex', 'justifyContent': 'space-around', 'marginBottom': '20px'}),

    html.Div(
        id='output-container',
        className='chart-grid',
        style={'display': 'flex', 'flexDirection': 'column'}
    ),
])

## 5. TASK 2.4 — Callbacks

We need two callbacks:

1. A callback that **enables/disables the Year dropdown** depending on which report type is
   selected (the Year dropdown only makes sense for "Yearly Statistics").
2. The main callback that **updates the output container** with the appropriate set of graphs
   based on the selected report type (and, for yearly statistics, the selected year).

In [14]:
@app.callback(
    Output('select-year', 'disabled'),
    Input('dropdown-statistics', 'value')
)
def update_input_container(selected_statistics):
    """TASK 2.4 (part 1): enable the Year dropdown only for 'Yearly Statistics'."""
    return selected_statistics != 'Yearly Statistics'


## 6. TASK 2.5 — Recession Report Statistics graphs

When **"Recession Period Statistics"** is selected, we display four charts built only from rows
where `Recession == 1`:

1. Average automobile sales over the years during recession (line chart).
2. Average number of vehicles sold by vehicle type during recession (bar chart).
3. Total expenditure share by vehicle type during recession (pie chart).
4. Effect of unemployment rate on vehicle type and sales during recession (bar chart).

In [15]:
def make_recession_figures():
    recession_data = df[df['Recession'] == 1]

    # 1. Average automobile sales over the years during recession
    yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
    fig1 = px.line(
        yearly_rec, x='Year', y='Automobile_Sales', markers=True,
        title='Average Automobile Sales over Years (Recession Periods)'
    )

    # 2. Average vehicles sold by vehicle type during recession
    avg_by_type = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
    fig2 = px.bar(
        avg_by_type, x='Vehicle_Type', y='Automobile_Sales',
        title='Average Vehicles Sold by Vehicle Type (Recession)'
    )

    # 3. Total advertising expenditure share by vehicle type during recession
    exp_by_type = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
    fig3 = px.pie(
        exp_by_type, names='Vehicle_Type', values='Advertising_Expenditure',
        title='Advertising Expenditure Share by Vehicle Type (Recession)'
    )

    # 4. Effect of unemployment rate on vehicle type and sales during recession
    unemp_effect = recession_data.groupby(['Vehicle_Type'], as_index=False).agg(
        Unemployment_Rate=('Unemployment_Rate', 'mean'),
        Automobile_Sales=('Automobile_Sales', 'mean')
    )
    fig4 = px.bar(
        unemp_effect, x='Vehicle_Type', y='Automobile_Sales', color='Unemployment_Rate',
        title='Unemployment Rate Effect on Sales by Vehicle Type (Recession)'
    )

    return fig1, fig2, fig3, fig4


## 7. TASK 2.6 — Yearly Report Statistics graphs

When **"Yearly Statistics"** is selected, we display four charts for the chosen year:

1. Yearly automobile sales trend across all years (line chart, whole timeline for context).
2. Total monthly automobile sales for the selected year (line chart).
3. Average vehicles sold by vehicle type in the selected year (bar chart).
4. Total advertising expenditure by vehicle type in the selected year (pie chart).

In [16]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

def make_yearly_figures(selected_year):
    # 1. Yearly automobile sales trend across the whole dataset (for context)
    yearly_all = df.groupby('Year')['Automobile_Sales'].mean().reset_index()
    fig1 = px.line(
        yearly_all, x='Year', y='Automobile_Sales', markers=True,
        title='Average Automobile Sales over Years (All Years)'
    )

    # 2. Total monthly automobile sales for the selected year
    year_data = df[df['Year'] == selected_year]
    monthly = year_data.groupby('Month')['Automobile_Sales'].sum().reindex(month_order).reset_index()
    fig2 = px.line(
        monthly, x='Month', y='Automobile_Sales', markers=True,
        title=f'Total Monthly Automobile Sales in {selected_year}'
    )

    # 3. Average vehicles sold by vehicle type in the selected year
    avg_by_type = year_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
    fig3 = px.bar(
        avg_by_type, x='Vehicle_Type', y='Automobile_Sales',
        title=f'Average Vehicles Sold by Vehicle Type in {selected_year}'
    )

    # 4. Total advertising expenditure by vehicle type in the selected year
    exp_by_type = year_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
    fig4 = px.pie(
        exp_by_type, names='Vehicle_Type', values='Advertising_Expenditure',
        title=f'Advertising Expenditure Share by Vehicle Type in {selected_year}'
    )

    return fig1, fig2, fig3, fig4

## 8. Main callback — wires everything together

This callback (the second half of TASK 2.4) listens to both dropdowns, decides which report to
build, and returns a populated `output-container` `<div>` containing the two rows of graphs from
TASK 2.5 / TASK 2.6.

In [17]:
@app.callback(
    Output('output-container', 'children'),
    [Input('dropdown-statistics', 'value'),
     Input('select-year', 'value')]
)
def update_output_container(selected_statistics, selected_year):
    if selected_statistics == 'Recession Period Statistics':
        fig1, fig2, fig3, fig4 = make_recession_figures()
    else:
        if selected_year is None:
            selected_year = year_list[-1]
        fig1, fig2, fig3, fig4 = make_yearly_figures(selected_year)

    row1 = html.Div(
        className='chart-item',
        children=[
            html.Div(dcc.Graph(figure=fig1), style={'width': '50%', 'display': 'inline-block'}),
            html.Div(dcc.Graph(figure=fig2), style={'width': '50%', 'display': 'inline-block'}),
        ],
        style={'display': 'flex'}
    )

    row2 = html.Div(
        className='chart-item',
        children=[
            html.Div(dcc.Graph(figure=fig3), style={'width': '50%', 'display': 'inline-block'}),
            html.Div(dcc.Graph(figure=fig4), style={'width': '50%', 'display': 'inline-block'}),
        ],
        style={'display': 'flex'}
    )

    return [row1, row2]

## 9. Run the app

Run the cell below to launch the dashboard.

- In **JupyterLab / classic Jupyter**, this opens an interactive server; use `mode='inline'`
  (requires `jupyter-dash`) or open the printed local URL in your browser.
- In a plain Python environment, simply run the script and visit `http://127.0.0.1:8050/`.

Use the **Select Statistics** dropdown to switch between "Recession Period Statistics" and
"Yearly Statistics", and the **Select Year** dropdown (enabled only for Yearly Statistics) to
drill into a specific year, exactly as requested by the directors.

In [20]:
import threading
import time
from google.colab import output

def run_app():
    app.run(debug=False, port=8050)

thread = threading.Thread(target=run_app)
thread.start()

time.sleep(3)  # give the server time to start
output.serve_kernel_port_as_iframe(8050, height=900)

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


<IPython.core.display.Javascript object>